In [25]:
import os 
import pandas as pd
import tensorflow as tf
import sionna
import pickle

# Set random seed for reproducibility
tf.random.set_seed(42)

# Commented out IPython magic to ensure Python compatibility.
import matplotlib.pyplot as plt
import numpy as np
import time
import json
import csv

# Import Sionna RT components
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, PathSolver, RadioMapSolver, subcarrier_frequencies

# For link-level simulations
from sionna.phy.channel import cir_to_ofdm_channel, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.phy.utils import ebnodb2no, PlotBER, dbm_to_watt, lin_to_db, log2, db_to_lin
from sionna.phy.ofdm import KBestDetector, LinearDetector, ResourceGrid
from sionna.phy.constants import BOLTZMANN_CONSTANT
from sionna.phy.mimo import StreamManagement

import copy

In [26]:
scene = load_scene("toy_XML/Chalmers_entrance.xml")

In [ ]:
x1=0.5
y1=-3
z1=15
tx1 = Transmitter("tx1", [x1,y1,z1], [0.0, 0.0, 0.0])   #Toymodel coordinates
scene.add(tx1)

#tx2 = Transmitter("tx2", [0.5, -343, 30], [0.0, 0.0, 0.0])   #Chalmers coordinates
#scene.add(tx2)

#tx3 = Transmitter("tx3", [65, -181.5, 35], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx3)

#tx4 = Transmitter("tx4", [22, 182, 28], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx4)

#tx5 = Transmitter("tx5", [-147, 327, 28], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx5)

#tx6 = Transmitter("tx6", [-212, -33, 15], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx6)
x2=50
y2=-3
z2=1
rx = Receiver("rx", [x2, y2, z2], [0.0, 0.0, 0.0])
scene.add(rx)   #toy example coordinates

#rx2 = Receiver("rx2", [-46, 28.5, 1.4], [0.0, 0.0, 0.0])
#scene.add(rx2)  #toy example coordinates

#rx3 = Receiver("rx3", [-74, 67, 1.6], [0.0, 0.0, 0.0])
#scene.add(rx3)  #toy example coordinates

#rx4 = Receiver("rx4", [1.3, 9, 1.4], [0.0, 0.0, 0.0])
#scene.add(rx4)  #toy example coordinates

#rx5 = Receiver("rx5", [70, -29, 1.6], [0.0, 0.0, 0.0])
#scene.add(rx5)  #toy example coordinates

#rx6 = Receiver("rx6", [-7.6, 28.5, 1.4], [0.0, 0.0, 0.0])
#scene.add(rx6)  #toy example coordinates

##rx8 = Receiver("rx8", [60, -294, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx8)

#rx9 = Receiver("rx9", [89, -169, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx9)

#rx10 = Receiver("rx10", [55.6, -26, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx10)

#rx11 = Receiver("rx11", [-35, 106, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx11)

#rx12 = Receiver("rx12", [-70, 0.3, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx12)

#rx13 = Receiver("rx13", [-116, 229, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx13)

#rx14 = Receiver("rx14", [-253, 402, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx14)

#rx15 = Receiver("rx15", [-235, 25, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx15)

#rx16 = Receiver("rx16", [-248, -52, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx16)


In [ ]:
# Set the carrier frequency 
fc = 3.5e9 # from band n78 in 3GPP TS 38.104. Can be changed later when testing out mmWave or sub-THz
scene.frequency = fc
# Define the wavelength of the signal and calculate the antenna distance 
c = 3e8
Lambda = c/fc
antenna_dist = Lambda/2

scene.tx_array = PlanarArray(num_rows=2, num_cols=2, vertical_spacing=0.5, horizontal_spacing=0.5,
                             pattern="iso", polarization="V")  # changed to cross-polarization as described in 3GPP TR 38.901
# Configure antenna array for all receivers
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=antenna_dist, horizontal_spacing=antenna_dist,
                             pattern="iso", polarization="V")
p_solver = PathSolver() 
# Compute the paths for the given receiver
paths = p_solver(scene=scene,
             max_depth=4,
             los=True,
             specular_reflection=True,
             diffuse_reflection=True,
             refraction=False,
             synthetic_array=True,
             seed=41)


In [29]:
scene.preview(paths=paths)

Renderer(camera=PerspectiveCamera(aspect=1.31, children=(DirectionalLight(intensity=0.25, position=(0.0, 0.0, …

In [30]:
#paths.theta_r    #referens i z-axel

In [31]:
#paths.theta_t

In [32]:
#paths.phi_r   #referens x-axel

In [33]:
#paths.phi_t

In [34]:
#paths.tau

In [35]:
#from math import sqrt

#path_delay_validation = sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2-z1)**2) / 299792458
#print(f"path delay validation: {path_delay_validation} s")

In [36]:
path_dict = paths.__dict__

print(path_dict)

{'_paths_buffer': <sionna.rt.path_solvers.paths_buffer.PathsBuffer object at 0x000002E7E54426B0>, '_src_positions': [[0.5, -3, 15]], '_tgt_positions': [[50, -3, 1]], '_tx_velocities': [[0, 0, 0]], '_rx_velocities': [[0, 0, 0]], '_tx_array': <sionna.rt.antenna_array.PlanarArray object at 0x000002E7E54423B0>, '_rx_array': <sionna.rt.antenna_array.PlanarArray object at 0x000002E7E5442D70>, '_num_tx': 1, '_num_rx': 1, '_synthetic_array': True, '_wavelength': [0.085655], '_frequency': [3.5e+09], '_paths_components_built': True, '_eff_tx_array_size': 1, '_eff_rx_array_size': 1, '_tx_ind': [0, 0, 0], '_rx_ind': [0, 0, 0], '_tx_ant_ind': [0, 0, 0], '_rx_ant_ind': [0, 0, 0], '_path_ind': [0, 1, 2], '_max_num_paths': 3, '_valid': [[[True, True, True]]], '_a_real': [[[[[2.21745e-05, 1.35246e-06, 1.35246e-06],
    [2.21745e-05, 1.78322e-06, 1.78322e-06],
    [2.21745e-05, 1.35246e-06, 1.35246e-06],
    [2.21745e-05, 1.78322e-06, 1.78322e-06]]]]], '_a_imag': [[[[[-1.01026e-05, -1.23303e-06, -1.2330